# 模型部署與生產環境
:label:`sec_deployment`

## 概述

訓練好模型只是深度學習項目的一半工作。將模型部署到生產環境，讓真實用戶能夠使用，是另一個關鍵挑戰。本節將介紹：

- 🔄 模型導出（ONNX, TorchScript）
- 🚀 模型服務化（TorchServe, FastAPI）
- 📱 邊緣設備部署
- ⚡ 移動端優化（iOS, Android）
- 🐳 容器化部署（Docker, Kubernetes）
- 🤖 AI輔助部署工具

## 1. 模型導出

### 1.1 TorchScript

TorchScript 是 PyTorch 模型的中間表示，可以在沒有 Python 依賴的環境中運行。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 定義一個簡單的模型
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.fc = nn.Linear(128 * 8 * 8, 10)
    
    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# 創建模型實例
model = SimpleNet()
model.eval()

# 方法1: Tracing（追蹤）
example_input = torch.randn(1, 3, 32, 32)
traced_model = torch.jit.trace(model, example_input)

# 保存模型
traced_model.save('model_traced.pt')

print("✅ TorchScript traced model saved")

In [ ]:
# 方法2: Scripting（腳本化）- 支持控制流
class ControlFlowNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 10)
    
    def forward(self, x):
        # 包含控制流，tracing無法處理
        if x.sum() > 0:
            return self.fc(x)
        else:
            return x * 2

model_with_control = ControlFlowNet()
scripted_model = torch.jit.script(model_with_control)
scripted_model.save('model_scripted.pt')

print("✅ TorchScript scripted model saved")

In [ ]:
# 加載和使用 TorchScript 模型
loaded_model = torch.jit.load('model_traced.pt')
loaded_model.eval()

# 推理
with torch.no_grad():
    output = loaded_model(example_input)
    print(f"Output shape: {output.shape}")
    print(f"Predictions: {torch.argmax(output, dim=1)}")

### 1.2 ONNX（Open Neural Network Exchange）

ONNX 是一個開放的模型格式，支持多種深度學習框架之間的互操作。

In [ ]:
# 導出為 ONNX 格式
model = SimpleNet()
model.eval()

dummy_input = torch.randn(1, 3, 32, 32)

# 導出模型
torch.onnx.export(
    model,                          # 模型
    dummy_input,                    # 示例輸入
    'model.onnx',                   # 輸出文件名
    export_params=True,             # 導出參數
    opset_version=11,               # ONNX opset版本
    do_constant_folding=True,       # 常量折疊優化
    input_names=['input'],          # 輸入名稱
    output_names=['output'],        # 輸出名稱
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print("✅ ONNX model exported")

In [ ]:
# 使用 ONNX Runtime 運行模型
# 首先安裝: pip install onnxruntime

try:
    import onnxruntime as ort
    import numpy as np
    
    # 創建推理會話
    ort_session = ort.InferenceSession('model.onnx')
    
    # 準備輸入
    ort_inputs = {'input': dummy_input.numpy()}
    
    # 運行推理
    ort_outputs = ort_session.run(None, ort_inputs)
    
    print(f"✅ ONNX Runtime inference successful")
    print(f"Output shape: {ort_outputs[0].shape}")
    
except ImportError:
    print("⚠️ ONNX Runtime not installed. Run: pip install onnxruntime")

## 2. 模型服務化

### 2.1 使用 FastAPI 構建 REST API

In [ ]:
# 保存以下代碼到 serve.py 文件
serve_code = '''
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
import torch
import torch.nn as nn
from PIL import Image
import io
from torchvision import transforms
import uvicorn

# 初始化 FastAPI
app = FastAPI(title="Image Classification API")

# 加載模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.jit.load("model_traced.pt", map_location=device)
model.eval()

# 預處理
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                       std=[0.229, 0.224, 0.225])
])

# 類別名稱
classes = ["airplane", "automobile", "bird", "cat", "deer", 
           "dog", "frog", "horse", "ship", "truck"]

@app.get("/")
async def root():
    return {"message": "Image Classification API is running"}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    try:
        # 讀取圖片
        image_bytes = await file.read()
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        
        # 預處理
        image_tensor = transform(image).unsqueeze(0).to(device)
        
        # 推理
        with torch.no_grad():
            output = model(image_tensor)
            probabilities = torch.softmax(output, dim=1)
            predicted_class = torch.argmax(probabilities, dim=1).item()
            confidence = probabilities[0][predicted_class].item()
        
        return JSONResponse(content={
            "class": classes[predicted_class],
            "confidence": float(confidence),
            "all_probabilities": {
                classes[i]: float(probabilities[0][i])
                for i in range(len(classes))
            }
        })
    
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"error": str(e)}
        )

@app.get("/health")
async def health():
    return {"status": "healthy"}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# 保存服務代碼
with open('serve.py', 'w') as f:
    f.write(serve_code)

print("✅ FastAPI server code saved to serve.py")
print("運行服務器: python serve.py")
print("API文檔: http://localhost:8000/docs")

### 2.2 客戶端測試代碼

In [ ]:
# 測試 API 的客戶端代碼
client_code = '''
import requests
from PIL import Image
import io

# API endpoint
url = "http://localhost:8000/predict"

# 準備圖片
image_path = "test_image.jpg"  # 替換為你的圖片路徑

with open(image_path, "rb") as f:
    files = {"file": ("image.jpg", f, "image/jpeg")}
    response = requests.post(url, files=files)

if response.status_code == 200:
    result = response.json()
    print(f"預測類別: {result['class']}")
    print(f"置信度: {result['confidence']:.2%}")
    print("\n所有類別的概率:")
    for class_name, prob in result['all_probabilities'].items():
        print(f"  {class_name}: {prob:.2%}")
else:
    print(f"錯誤: {response.status_code}")
    print(response.json())
'''

with open('client.py', 'w') as f:
    f.write(client_code)

print("✅ Client code saved to client.py")

## 3. 容器化部署

### 3.1 Dockerfile

In [ ]:
# Dockerfile 內容
dockerfile_content = '''
FROM python:3.10-slim

WORKDIR /app

# 安裝依賴
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 複製模型和代碼
COPY model_traced.pt .
COPY serve.py .

# 暴露端口
EXPOSE 8000

# 健康檢查
HEALTHCHECK --interval=30s --timeout=30s --start-period=5s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

# 運行服務
CMD ["python", "serve.py"]
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile_content)

# requirements.txt
requirements = '''torch==2.1.0
torchvision==0.16.0
fastapi==0.104.1
uvicorn[standard]==0.24.0
python-multipart==0.0.6
pillow==10.1.0
'''

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("✅ Dockerfile and requirements.txt created")
print("\n構建 Docker 鏡像:")
print("  docker build -t model-api:v1 .")
print("\n運行容器:")
print("  docker run -p 8000:8000 model-api:v1")

### 3.2 Docker Compose

In [ ]:
# docker-compose.yml
docker_compose = '''
version: '3.8'

services:
  model-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/model_traced.pt
    volumes:
      - ./logs:/app/logs
    restart: unless-stopped
    deploy:
      resources:
        limits:
          cpus: '2'
          memory: 4G

  nginx:
    image: nginx:alpine
    ports:
      - "80:80"
    volumes:
      - ./nginx.conf:/etc/nginx/nginx.conf:ro
    depends_on:
      - model-api
    restart: unless-stopped
'''

with open('docker-compose.yml', 'w') as f:
    f.write(docker_compose)

print("✅ docker-compose.yml created")
print("\n啟動服務:")
print("  docker-compose up -d")

## 4. 邊緣設備部署

### 4.1 模型優化

In [ ]:
# 量化模型以減小體積和提高推理速度
from torch.quantization import quantize_dynamic

model = SimpleNet()
model.eval()

# 動態量化
quantized_model = quantize_dynamic(
    model,
    {nn.Linear, nn.Conv2d},  # 要量化的層類型
    dtype=torch.qint8
)

# 比較模型大小
import os

torch.save(model.state_dict(), 'model_fp32.pth')
torch.save(quantized_model.state_dict(), 'model_int8.pth')

fp32_size = os.path.getsize('model_fp32.pth') / 1024  # KB
int8_size = os.path.getsize('model_int8.pth') / 1024  # KB

print(f"原始模型大小: {fp32_size:.2f} KB")
print(f"量化模型大小: {int8_size:.2f} KB")
print(f"壓縮比: {fp32_size/int8_size:.2f}x")

### 4.2 Raspberry Pi 部署示例

In [ ]:
# 輕量級推理代碼（適用於樹莓派等設備）
raspberry_pi_code = '''
import torch
import torch.nn as nn
from PIL import Image
import time

class LightweightInference:
    def __init__(self, model_path):
        # 加載量化模型
        self.model = torch.jit.load(model_path)
        self.model.eval()
        
    def preprocess(self, image_path):
        """輕量級預處理"""
        image = Image.open(image_path).convert('RGB')
        image = image.resize((32, 32))
        # 簡化的預處理
        import numpy as np
        image_array = np.array(image).astype(np.float32) / 255.0
        image_tensor = torch.from_numpy(image_array).permute(2, 0, 1).unsqueeze(0)
        return image_tensor
    
    def predict(self, image_path):
        """執行推理"""
        start_time = time.time()
        
        # 預處理
        image_tensor = self.preprocess(image_path)
        
        # 推理
        with torch.no_grad():
            output = self.model(image_tensor)
            predicted_class = torch.argmax(output, dim=1).item()
        
        inference_time = (time.time() - start_time) * 1000  # ms
        
        return predicted_class, inference_time

# 使用示例
if __name__ == "__main__":
    inferencer = LightweightInference("model_quantized.pt")
    
    # 測試推理
    predicted_class, inference_time = inferencer.predict("test.jpg")
    print(f"Predicted class: {predicted_class}")
    print(f"Inference time: {inference_time:.2f} ms")
'''

with open('raspberry_pi_inference.py', 'w') as f:
    f.write(raspberry_pi_code)

print("✅ Raspberry Pi inference code saved")

## 5. 移動端部署

### 5.1 PyTorch Mobile (iOS/Android)

In [ ]:
# 為移動端優化模型
from torch.utils.mobile_optimizer import optimize_for_mobile

model = SimpleNet()
model.eval()

# Trace模型
example_input = torch.randn(1, 3, 32, 32)
traced_model = torch.jit.trace(model, example_input)

# 移動端優化
optimized_model = optimize_for_mobile(traced_model)

# 保存為移動端格式
optimized_model._save_for_lite_interpreter("model_mobile.ptl")

print("✅ Mobile-optimized model saved")
print("\n集成到iOS/Android:")
print("iOS: 使用 PyTorch iOS SDK")
print("Android: 使用 PyTorch Android SDK")

### 5.2 iOS 集成示例（Swift）

In [ ]:
# iOS Swift代碼示例
ios_swift_code = '''
import UIKit
import CoreML

class ImageClassifier {
    private var module: TorchModule?
    
    init() {
        // 加載模型
        if let filePath = Bundle.main.path(forResource: "model_mobile", ofType: "ptl") {
            module = TorchModule(fileAtPath: filePath)
        }
    }
    
    func predict(image: UIImage) -> String? {
        guard let module = module else { return nil }
        
        // 預處理圖片
        guard let pixelBuffer = image.pixelBuffer(width: 32, height: 32) else {
            return nil
        }
        
        // 轉換為Tensor
        let inputTensor = TorchTensor(pixelBuffer: pixelBuffer)
        
        // 推理
        guard let outputTensor = module.forward(inputTensor) else {
            return nil
        }
        
        // 獲取預測結果
        let scores = outputTensor.floatArray()
        let predictedIndex = scores.enumerated().max(by: { $0.element < $1.element })?.offset
        
        let classes = ["airplane", "automobile", "bird", "cat", "deer", 
                       "dog", "frog", "horse", "ship", "truck"]
        
        return predictedIndex.map { classes[$0] }
    }
}
'''

with open('ImageClassifier.swift', 'w') as f:
    f.write(ios_swift_code)

print("✅ iOS Swift code saved")

## 6. AI輔助部署工具

### 6.1 使用 Hugging Face Hub 分享模型

In [ ]:
# 上傳模型到 Hugging Face Hub
hf_hub_code = '''
from huggingface_hub import HfApi, create_repo
import torch

# 初始化 API
api = HfApi()

# 創建倉庫
repo_id = "your-username/your-model-name"
create_repo(repo_id, exist_ok=True)

# 上傳模型文件
api.upload_file(
    path_or_fileobj="model_traced.pt",
    path_in_repo="model.pt",
    repo_id=repo_id,
    repo_type="model"
)

# 上傳配置文件
config = {
    "model_type": "image_classification",
    "input_size": [3, 32, 32],
    "num_classes": 10
}

import json
with open("config.json", "w") as f:
    json.dump(config, f)

api.upload_file(
    path_or_fileobj="config.json",
    path_in_repo="config.json",
    repo_id=repo_id,
    repo_type="model"
)

print(f"✅ Model uploaded to: https://huggingface.co/{repo_id}")
'''

print(hf_hub_code)
print("\n安裝 Hugging Face Hub:")
print("  pip install huggingface-hub")

### 6.2 使用 Gradio 快速創建演示

In [ ]:
# Gradio 演示代碼
gradio_code = '''
import gradio as gr
import torch
from PIL import Image
import numpy as np

# 加載模型
model = torch.jit.load("model_traced.pt")
model.eval()

classes = ["airplane", "automobile", "bird", "cat", "deer", 
           "dog", "frog", "horse", "ship", "truck"]

def predict(image):
    # 預處理
    image = Image.fromarray(image.astype('uint8'), 'RGB')
    image = image.resize((32, 32))
    image_array = np.array(image).astype(np.float32) / 255.0
    image_tensor = torch.from_numpy(image_array).permute(2, 0, 1).unsqueeze(0)
    
    # 推理
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.softmax(output, dim=1)[0]
    
    # 返回預測結果
    return {classes[i]: float(probabilities[i]) for i in range(len(classes))}

# 創建界面
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(),
    outputs=gr.Label(num_top_classes=3),
    title="Image Classification",
    description="Upload an image to classify it into one of 10 categories.",
    examples=[
        ["example1.jpg"],
        ["example2.jpg"],
    ]
)

# 啟動
demo.launch(share=True)  # share=True 生成公開鏈接
'''

with open('gradio_demo.py', 'w') as f:
    f.write(gradio_code)

print("✅ Gradio demo code saved")
print("\n運行演示:")
print("  pip install gradio")
print("  python gradio_demo.py")

## 7. 生產環境最佳實踐

### 7.1 監控和日誌

In [ ]:
# 添加監控和日誌的完整服務代碼
production_serve_code = '''
from fastapi import FastAPI, File, UploadFile
from prometheus_client import Counter, Histogram, make_asgi_app
import logging
import time

# 配置日誌
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('logs/api.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Prometheus 指標
REQUEST_COUNT = Counter('request_count', 'Total request count', ['method', 'endpoint'])
REQUEST_LATENCY = Histogram('request_latency_seconds', 'Request latency')
PREDICTION_COUNT = Counter('prediction_count', 'Total predictions', ['class'])

app = FastAPI()

# 添加 Prometheus metrics endpoint
metrics_app = make_asgi_app()
app.mount("/metrics", metrics_app)

@app.middleware("http")
async def add_monitoring(request, call_next):
    start_time = time.time()
    
    REQUEST_COUNT.labels(method=request.method, endpoint=request.url.path).inc()
    
    response = await call_next(request)
    
    process_time = time.time() - start_time
    REQUEST_LATENCY.observe(process_time)
    
    logger.info(f"{request.method} {request.url.path} - {response.status_code} - {process_time:.3f}s")
    
    return response

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    try:
        # ... (推理代碼)
        
        PREDICTION_COUNT.labels(class_name=predicted_class).inc()
        
        return {"class": predicted_class, "confidence": confidence}
    
    except Exception as e:
        logger.error(f"Prediction error: {str(e)}", exc_info=True)
        raise
'''

print(production_serve_code)

### 7.2 A/B 測試

In [ ]:
# A/B 測試代碼
ab_testing_code = '''
import random
from enum import Enum

class ModelVersion(Enum):
    V1 = "model_v1.pt"
    V2 = "model_v2.pt"

class ABTestingRouter:
    def __init__(self, v1_weight=0.5):
        self.v1_weight = v1_weight
        self.models = {
            ModelVersion.V1: torch.jit.load(ModelVersion.V1.value),
            ModelVersion.V2: torch.jit.load(ModelVersion.V2.value)
        }
    
    def get_model(self, user_id=None):
        """根據用戶ID或隨機選擇模型版本"""
        if user_id:
            # 基於用戶ID的一致性路由
            hash_value = hash(user_id) % 100
            version = ModelVersion.V1 if hash_value < self.v1_weight * 100 else ModelVersion.V2
        else:
            # 隨機路由
            version = ModelVersion.V1 if random.random() < self.v1_weight else ModelVersion.V2
        
        return self.models[version], version

# 使用示例
router = ABTestingRouter(v1_weight=0.7)  # 70% 用戶使用 V1

@app.post("/predict")
async def predict(file: UploadFile, user_id: str = None):
    model, version = router.get_model(user_id)
    
    # 執行推理...
    result = model(image_tensor)
    
    # 記錄使用的模型版本
    logger.info(f"Used model version: {version.name} for user: {user_id}")
    
    return {"result": result, "model_version": version.name}
'''

print(ab_testing_code)

## 小結

本章介紹了深度學習模型部署的完整流程：

✅ **模型導出**：TorchScript、ONNX
✅ **服務化**：FastAPI、TorchServe
✅ **容器化**：Docker、Kubernetes
✅ **邊緣部署**：Raspberry Pi、嵌入式設備
✅ **移動端**：iOS、Android
✅ **AI輔助工具**：Hugging Face Hub、Gradio
✅ **生產環境**：監控、日誌、A/B測試

## 練習

1. 將你訓練的模型部署為 REST API
2. 使用 Docker 容器化你的服務
3. 創建一個 Gradio 演示界面
4. 實現模型的 A/B 測試
5. 添加 Prometheus 監控指標

## 推薦資源

- 📖 [TorchServe Documentation](https://pytorch.org/serve/)
- 📖 [FastAPI Documentation](https://fastapi.tiangolo.com/)
- 📖 [ONNX Runtime](https://onnxruntime.ai/)
- 📖 [Gradio Documentation](https://gradio.app/)
- 📖 [PyTorch Mobile](https://pytorch.org/mobile/home/)